In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns
import pandas as pd
import hydra
import lightning as lit
import torch
import torch.nn as nn
import wandb
import os
import palettable
import numpy as np
from tqdm.auto import tqdm
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from datetime import datetime

from machine_intelligence.utilities.viz_setup import figure_setup_grid, VariableNames

from machine_intelligence.utilities.viz_setup import figure_setup, five_min_to_hours
import matplotlib.ticker as ticker
import random

from machine_intelligence.models.GatedTransformer import GatedTransformer
from machine_intelligence.data.SepsisPredDataModule import SepsisPredDataModule



In [ ]:

cwd = Path.cwd()
cwd

In [ ]:

project_root = Path(cwd).parents[1]
project_root

In [ ]:

current_date = datetime.now().strftime('%Y-%m-%d')
plot_output_dir = project_root / 'plot_outputs' / current_date
plot_output_dir.mkdir(parents=True, exist_ok=True)
# model_path = project_root / Path("experiment_logs/gtn-debug/version_118/checkpoints/epoch=5-step=186.ckpt")

In [ ]:

model_path = project_root / Path("machine_intelligence/models/trained_models/gtn_final-3/checkpoints/last.ckpt")
model = GatedTransformer.load_from_checkpoint(model_path, map_location="cpu")


In [ ]:

wandb.init(mode="disabled")


In [ ]:

os.environ['PYTHONHASHSEED'] = str(42)
lit.seed_everything(42, workers=True)
torch.use_deterministic_algorithms(True)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


In [ ]:
with initialize(version_base=None, config_path="../../configs"):
    cfg = compose(config_name="run_training.yaml", overrides=["data=cohort_a_r0.4_prev_0.3_imp_none"])

    print(cfg)


In [ ]:
sepsis_pred_data = SepsisPredDataModule(cfg=cfg)

sepsis_pred_data.setup(stage="fit")
sepsis_pred_data.setup(stage="test")

train_dl = sepsis_pred_data.train_dataloader()
test_dl = sepsis_pred_data.test_dataloader()

# batch = next(iter(test_dl))
# batch_features_ts, batch_features_stat, batch_labels = batch


In [ ]:
score_channel_averaged_across_test = []

model.to("cpu")
model.eval()

with torch.no_grad():
    for batch in tqdm(test_dl, total=len(test_dl)):
        sample_ts, sample_stat, sample_label = batch  # Adjust according to your dataloader's output

        pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
            sample_ts, sample_stat, stage="test")

        # batch mean
        score_feature_mean = torch.mean(score_channel_sm, dim=0)

        score_channel_averaged_across_test.append(score_feature_mean)



In [ ]:
score_channel_averaged_across_train = []

model.to("cpu")
model.eval()

with torch.no_grad():
    for batch in tqdm(train_dl, total=len(train_dl)):
        sample_ts, sample_stat, sample_label = batch  # Adjust according to your dataloader's output

        pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
            sample_ts, sample_stat, stage="test")

        # batch mean
        score_feature_mean = torch.mean(score_channel_sm, dim=0)

        score_channel_averaged_across_train.append(score_feature_mean)


In [ ]:
all_score_channel_average_train = torch.mean(torch.stack(score_channel_averaged_across_train), dim=0)
all_score_channel_average_test = torch.mean(torch.stack(score_channel_averaged_across_test), dim=0)

In [ ]:
all_score_channel_average_train

In [ ]:
all_score_channel_average_test

Not exactly the same but, every distinction seems to be averaged out

In [ ]:
len(score_input)

In [ ]:
score_input[0].shape

The gtn has only one layer so we can access it directly the outputs from the forward pass above!

In [ ]:
model.score_input_list[0].shape

### Softmaxed Feature Level Attention Averaged across heads

In [ ]:
df_score_channel_mean = pd.DataFrame(data=all_score_channel_average_test,
                                     columns=sepsis_pred_data.considered_features,
                                     index=sepsis_pred_data.considered_features)

df_score_channel_mean = VariableNames.get_pretty_dataframe(df=df_score_channel_mean)
df_score_channel_mean = VariableNames.get_human_readable_index(df=df_score_channel_mean)
df_score_channel_mean = VariableNames.sort_dataframe(df=df_score_channel_mean)

fig, axs, font_labels, font_annotations = figure_setup(1, 1, width_cm=15, height_cm=12, num_colors=11,
                                                       height_ratios=[1], share_y=False, share_x=False)

# sns.heatmap(df_score_ch_top20, ax=axs, cbar=True, cbar_kws={"label": "Attention Score"})
# sns.heatmap(df_score_channel_sm_mean, ax=axs, cbar=True, cbar_kws={"label": "Attention Score"})
sns.heatmap(df_score_channel_mean, ax=axs, square=True, cbar=True, cbar_kws={"label": "Attention Score"},
            xticklabels=1, yticklabels=1, cmap=sns.color_palette("magma", as_cmap=True))

# axs.invert_xaxis()
# axs.invert_yaxis()

axs.set_xlabel("Time Series Variables",
               )

axs.set_ylabel("Time Series Variables",
               )

axs.set_xticklabels(axs.get_xticklabels(), fontsize=3, rotation=80, ha="right")
axs.set_yticklabels(axs.get_yticklabels(), fontsize=3)

# Adjust layout
plt.tight_layout()
plt.show()

if save_as_svg := True:
    fig.savefig(plot_output_dir / Path(f"global_attention_features_averaged_test.svg"), format="svg")
    fig.savefig(plot_output_dir / Path(f"global_attention_features_averaged_test.png"), format="png", dpi=300)



In [ ]:
df_score_channel_mean_train = pd.DataFrame(data=all_score_channel_average_train,
                                           columns=sepsis_pred_data.considered_features,
                                           index=sepsis_pred_data.considered_features)

df_score_channel_mean_train = VariableNames.get_pretty_dataframe(df=df_score_channel_mean_train)
df_score_channel_mean_train = VariableNames.get_human_readable_index(df=df_score_channel_mean_train)
df_score_channel_mean_train = VariableNames.sort_dataframe(df=df_score_channel_mean_train)

fig, axs, font_labels, font_annotations = figure_setup(1, 1, width_cm=15, height_cm=12, num_colors=11,
                                                       height_ratios=[1], share_y=False, share_x=False)

# sns.heatmap(df_score_ch_top20, ax=axs, cbar=True, cbar_kws={"label": "Attention Score"})
# sns.heatmap(df_score_channel_sm_mean, ax=axs, cbar=True, cbar_kws={"label": "Attention Score"})
sns.heatmap(df_score_channel_mean_train, ax=axs, square=True, cbar=True, cbar_kws={"label": "Attention Score"},
            xticklabels=1, yticklabels=1, cmap=sns.color_palette("magma", as_cmap=True))

# axs.invert_xaxis()
# axs.invert_yaxis()

axs.set_xlabel("Time Series Variables",
               )

axs.set_ylabel("Time Series Variables",
               )

axs.set_xticklabels(axs.get_xticklabels(), fontsize=3, rotation=80, ha="right")
axs.set_yticklabels(axs.get_yticklabels(), fontsize=3)

# Adjust layout
plt.tight_layout()
plt.show()

if save_as_svg := True:
    fig.savefig(plot_output_dir / Path(f"global_attention_features_averaged_train.svg"), format="svg")
    fig.savefig(plot_output_dir / Path(f"global_attention_features_averaged_train.png"), format="png",dpi=300)




In [ ]:
attention_score_channel_mean_train_query = torch.mean(all_score_channel_average_train, axis=0)
attention_score_channel_mean_train_key = torch.mean(all_score_channel_average_train, axis=1)


attention_score_channel_mean_test_query = torch.mean(all_score_channel_average_test, axis=0)
attention_score_channel_mean_test_key = torch.mean(all_score_channel_average_test, axis=1)



In [ ]:
attention_score_channel_mean_train_query.shape

In [ ]:
df_train_query = pd.DataFrame(data=attention_score_channel_mean_train_query.unsqueeze(1).T,
                                           columns=sepsis_pred_data.considered_features,
                                           # index=sepsis_pred_data.considered_features,
                              )

df_test_query = pd.DataFrame(data=attention_score_channel_mean_test_query.unsqueeze(1).T,
                                           columns=sepsis_pred_data.considered_features,
                                           # index=sepsis_pred_data.considered_features,
                              )


In [ ]:

fig, axs, font_labels, font_annotations = figure_setup(1, 1, width_cm=15, height_cm=12, num_colors=11,
                                                       height_ratios=[1], share_y=False, share_x=False)

# sns.barplot(df_train_query, 
sns.heatmap(df_train_query, 
            xticklabels=1,
            ax=axs)


axs.set_xticklabels(axs.get_xticklabels(), fontsize=3, rotation=80, ha="right")
axs.set_yticklabels(axs.get_yticklabels(), fontsize=3)


In [ ]:
fig, axs, font_labels, font_annotations = figure_setup(1, 1, width_cm=15, height_cm=12, num_colors=11,
                                                       height_ratios=[1], share_y=False, share_x=False)
# sns.barplot(df_test_query,
sns.heatmap(df_test_query,
            xticklabels=1, 
            ax=axs,
            cmap=sns.color_palette("magma", as_cmap=True))

axs.set_xticklabels(axs.get_xticklabels(), fontsize=3, rotation=80, ha="right")
axs.set_yticklabels(axs.get_yticklabels(), fontsize=3)


In [ ]:
sns.barplot(df_test_query)